In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchinfo import summary
from torchviz import make_dot
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.datasets as datasets
import sys
import os

In [2]:
# Import custom module
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

%load_ext autoreload
%autoreload 2

from modules.Contrastive_Divergence_Minimisation import InteractionSufficientStatistics, ExponentialFamilyModel
# from modules.Annealed_Importance_Sampling import 

In [3]:
# set devise
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# prepare MNIST data
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(root='./data', train=True, download=False, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=False, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

Using device: cpu


/cs/student/msc/csml/2025/katayama/.local/lib/python3.9/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


In [8]:
# define 28*28 -> 50 -> 10 -> 5 -> 5 -> 5 -> 10 network
class MNISTNet(nn.Module):
    def __init__(self):
        super(MNISTNet, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 50)
        self.fc2 = nn.Linear(50, 10)
        self.fc3 = nn.Linear(10, 5)
        self.fc4 = nn.Linear(5, 5)
        self.fc5 = nn.Linear(5, 5)
        self.fc6 = nn.Linear(5, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = torch.relu(self.fc5(x))
        x = self.fc6(x)
        return x
    
    def get_activations(self, x):
        x = self.flatten(x)
        act1 = torch.relu(self.fc1(x))
        act2 = torch.relu(self.fc2(act1))
        act3 = torch.relu(self.fc3(act2))
        act4 = torch.relu(self.fc4(act3))
        act5 = torch.relu(self.fc5(act4))
        act6 = self.fc6(act5)

        _, predicted = torch.max(act6, 1)

        return {
            'L1': act1.detach().cpu().numpy(),
            'L2': act2.detach().cpu().numpy(),
            'L3': act3.detach().cpu().numpy(),
            'L4': act4.detach().cpu().numpy(),
            'L5': act5.detach().cpu().numpy(),
            'L6': act6.detach().cpu().numpy(),
            'predicted': predicted.detach().cpu().numpy()
        }

In [13]:
# initialise model
model = MNISTNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
max_order = 5 # 組み合わせ爆発を防ぐための打ち切り次数
calc_stats = InteractionSufficientStatistics(max_order=max_order)

epochs = 100 
cd_steps_per_epoch = 10

for epoch in range(epochs):
    # ==========================================
    # 1. learning phase
    # ==========================================
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")
    
    # ==========================================
    # 2. obtain activation
    # ==========================================
    l1, l2, l3, l4, l5, l6 = [], [], [], [], [], []
    predictions = []
    
    model.eval() 
    with torch.no_grad(): 
        for images, labels in test_loader:
            images = images.to(device)
            
            activations = model.get_activations(images)
            l1.append(activations['L1'])
            l2.append(activations['L2'])
            l3.append(activations['L3'])
            l4.append(activations['L4'])
            l5.append(activations['L5'])
            l6.append(activations['L6'])
            predictions.append(activations['predicted'])
            
    l1 = np.vstack(l1)
    print(l1)
    l2 = np.vstack(l2)
    l3 = np.vstack(l3)
    l4 = np.vstack(l4)
    l5 = np.vstack(l5)
    l6 = np.vstack(l6)
    predictions = np.concatenate(predictions)
    
    # ==========================================
    # 3. Information Geometry / CDM Phase
    # ==========================================
    S_all = l1  # NumPy配列のまま保持 (ここではまだGPUに乗せない)
    T_all = predictions
    num_samples = len(S_all)
    
    cdm_batch_size = 512
    
    global_pos_mean = {k: 0.0 for k in range(max_order + 1)}
    
    # ミニバッチごとに計算して合計していく
    with torch.no_grad():
        for i in range(0, num_samples, cdm_batch_size):
            S_batch = torch.tensor(S_all[i:i+cdm_batch_size], dtype=torch.float32).to(device)
            T_batch = torch.tensor(T_all[i:i+cdm_batch_size], dtype=torch.long).to(device)
            T_onehot_batch = F.one_hot(T_batch, num_classes=10).float()
            
            pos_stats_batch = calc_stats(T_onehot_batch, S_batch)
            pos_stats_batch[0] = T_onehot_batch
            
            for k in range(max_order + 1):
                global_pos_mean[k] += pos_stats_batch[k].sum(dim=0)
                
        # 最後に全体のデータ数で割って「真の期待値」にする
        for k in range(max_order + 1):
            global_pos_mean[k] /= num_samples

    # ----------------------------------------------------
    # Phase B: モデルの初期化と 0次の解析的フィッティング
    # ----------------------------------------------------
    cdm_models = {}
    cdm_optimizers = {}
    
    stats_dim_0 = global_pos_mean[0].shape[0]
    cdm_models[0] = ExponentialFamilyModel(stats_dim=stats_dim_0).to(device)
    
    with torch.no_grad():
        eps = 1e-8
        cdm_models[0].theta.copy_(torch.log(global_pos_mean[0] + eps))
    
    # ----------------------------------------------------
    # Phase C: k=1 から max_order までの最適化 (SGD)
    # ----------------------------------------------------
    # インナーループのエポック数（モデルが収束するまでの周回数）
    cd_epochs = 10 
    
    for k in range(1, max_order + 1):
        stats_dim = global_pos_mean[k].shape[0]
        cdm_models[k] = ExponentialFamilyModel(stats_dim=stats_dim).to(device)
        
        if k > 0:
            with torch.no_grad():
                prev_dim = cdm_models[k-1].theta.shape[0]
                cdm_models[k].theta[:prev_dim].copy_(cdm_models[k-1].theta)
                
        cdm_optimizers[k] = torch.optim.SGD(cdm_models[k].parameters(), lr=0.01)
        
        # パラメータ収束のための学習ループ
        for inner_epoch in range(cd_epochs):
            
            # データをシャッフルしてSGDを行う
            indices = np.random.permutation(num_samples)
            
            for i in range(0, num_samples, cdm_batch_size):
                idx = indices[i:i+cdm_batch_size]
                S_batch = torch.tensor(S_all[idx], dtype=torch.float32).to(device)
                
                # 1. 経験分布 S_batch を元に、モデルの期待値を解析的に計算
                neg_mean_batch = cdm_models[k].compute_negative_stats_mean(
                    S_batch=S_batch, 
                    calc_stats_fn=calc_stats, 
                    k=k
                )
                
                # 2. 実データの真の期待値 (global_pos_mean) と比較してパラメータを更新
                loss_val = cdm_models[k].contrastive_divergence_step(
                    optimizer=cdm_optimizers[k], 
                    pos_mean=global_pos_mean[k], # ターゲットは常にブレない全体期待値
                    neg_mean=neg_mean_batch
                )

Epoch [1/100], Loss: 1.3326
[[ 8.794316    3.320472    1.302552   ...  0.          1.0790544
   0.        ]
 [11.034689    2.4142997  10.965172   ...  0.          3.593398
   5.2865667 ]
 [ 7.350346    0.5758674   3.123842   ...  1.1566253   0.4944445
   1.7141103 ]
 ...
 [13.904385    0.7281586   2.4648788  ...  0.          0.07207471
   3.0392978 ]
 [ 8.05742     0.          5.864229   ...  0.          1.4895406
   3.608746  ]
 [ 8.180167    0.          5.0122995  ...  8.1224375   0.
   6.885067  ]]


: 

In [16]:
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"\nAccuracy on the test data: {100 * correct / total:.2f}%")


Accuracy on the test data: 95.47%
